# Textual Entailment Fact Checking

Use a pre-trained DeBERTa-v3 NLI model (MNLI + FEVER + ANLI) to decide whether each fact (hypothesis) is entailed by a passage sentence (premise). The three-way output (entailment / neutral / contradiction) is mapped to S / NS. Target should be more than or equal to 83% accuracy.

**Plan is to** run the model once over every (sentence, fact) pair -> cache the probabilities -> tune decision rules, thresholds and pruning offline.

## 1. Imports and paths

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Project root (two levels up) so we can import factcheck.py and factchecking_main.py directly
ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

# Reuse the official loaders
from factchecking_main import read_passages, read_fact_examples
# Reuse the Part 1 word-overlap scorer for pruning.
from factcheck import WordRecallThresholdFactChecker

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

/Users/seyedrumaiz/Library/Mobile Documents/com~apple~CloudDocs/University/Year 3/CM3604/LLMFactChecking/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load data (loaders of factchecking_main.py)

In [ ]:
# Load the data and convert to binary labels (S vs NS, where NS includes IR)
fact_to_passage = read_passages(str(ROOT / "data/passages_bm25_ChatGPT_humfacts.jsonl"))
examples = read_fact_examples(str(ROOT / "data/dev_labeled_ChatGPT.jsonl"), fact_to_passage)

# IR -> NS, exactly like predict_two_classes() in factchecking_main.py
y_true = np.array([1 if ex.label == "S" else 0 for ex in examples])

print(f"Examples: {len(examples)}  |  S = {y_true.sum()}  |  NS (incl. IR) = {(y_true == 0).sum()}")

Examples: 221  |  S = 112  |  NS (incl. IR) = 109


## 3. Load the entailment model

In [3]:
MODEL_NAME = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

# Confirm the label order instead of assuming it.
print(model.config.id2label)

{0: 'entailment', 1: 'neutral', 2: 'contradiction'}
